# 18.7 教師需要跟著更新嗎？


蒸餾時若教師也跟學生誤差一起更新，學生追的目標可能每一步移動，就不再是固定教師提供的指導。本章使用固定教師：先選已驗證有能力的權重，教師產生訊號，學生更新自己的數字。前置是[梯度與更新](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)和[eval與梯度](https://birdhackor.github.io/tiny-perceptron-vlm/5.17.html)。這裡要分清三個工具，各自負責的事不同。

`eval()`切換模組行為，例如停用訓練時的dropout，不會自動停用導數記錄；`requires_grad_(False)`讓教師權重不求梯度；`no_grad()`則讓該區塊的教師計算不建立反向運算圖。優化器只接收學生參數，決定更新哪些數字。一起使用能清楚表達固定教師的角色，也避免保存不需要的教師中間計算。

下面用兩個很小的線性層，只檢驗固定與更新；這個教師隨機初始化，沒有已學會的知識。輸入 `[1,2]`，教師和學生都輸出三個特徵，用兩者平方差平均當學生誤差，方便檢查一次更新是否只改學生。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
teacher = nn.Linear(2, 3).eval().requires_grad_(False)
student = nn.Linear(2, 3)
optimizer = torch.optim.SGD(student.parameters(), lr=0.1)
x = torch.tensor([[1.0, 2.0]])
teacher_before = teacher.weight.detach().clone()
student_before = student.weight.detach().clone()
with torch.no_grad():
    target = teacher(x)
output = student(x)
optimizer.zero_grad()
(output - target).square().mean().backward()
optimizer.step()
print("教師梯度", teacher.weight.grad)
print("教師最大改動", (teacher.weight - teacher_before).abs().max().item())
print("學生有改動", bool((student.weight - student_before).abs().max() > 0))
probe = x.clone().requires_grad_()
print("輸入可微時教師輸出記圖", teacher(probe).requires_grad)

應印出教師梯度None、最大改動0、學生有改動True。`SGD`是沿梯度反方向更新的優化器，lr=0.1表示步幅；它只拿student.parameters，所以沒有教師權重。快照的clone建立獨立數值副本，才能比較更新前後；若只保留引用，檢查就會同時看到新權重而失效。

最後一行應為True：即使教師權重凍結，probe輸入需要梯度，普通teacher(probe)仍會記錄如何從輸入影響輸出。這解釋為何no_grad與freeze不是完全相同的事。正式蒸餾通常不希望通過教師教輸入端，所以教師生成目標那段仍使用no_grad；若另設可微教師用途，應明確說明改變了哪條學習路徑。

凍結不等於教師計算免費，forward仍要讀權重、做矩陣運算；若預先保存目標可以重用，也會增加儲存與版本管理成本。教師的能力應由獨立任務驗證，不是由 `grad is None`判斷。

練習只把最後 `teacher(probe)`也包進no_grad，再預測requires_grad變False，執行核對。其他訓練更新那段不用改，教師仍不變、學生仍更新；你就能具體分辨模組模式、權重凍結與整段不記圖三種機制。
